##Persiapan Data & Tokenizer SOTA

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics.pairwise import cosine_similarity
import json, math, random, datetime

# Load TensorBoard untuk Google Colab (Quest 7)
%load_ext tensorboard

# 1. Load Data
df = pd.read_excel('dataset_resep_format_final.xlsx')

# Trik SOTA: Gabungkan nama menu dan bahan agar AI paham metode masak
df['teks_kunci_sota'] = df['nama_menu'].astype(str) + " " + df['bahan'].astype(str)

# 2. Inisialisasi Tokenizer Keras
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(df['teks_kunci_sota'])

VOCAB_SIZE = len(tokenizer.word_index) + 1
MAX_LEN = 30
EMBED_DIM = 64
TOWER_DIM = 32

# 3. Ubah resep ke Sequence Angka
X_resep_seq = tokenizer.texts_to_sequences(df['teks_kunci_sota'])
X_resep = pad_sequences(X_resep_seq, maxlen=MAX_LEN, padding='post')

# 4. Pembangkitan Data Latih (Positive & Negative Pairs)
X1_list, X2_list, labels_list = [], [], []
num_recipes = len(X_resep)

for _ in range(10000):
    idx_resep = random.randint(0, num_recipes - 1)
    resep_asli = X_resep[idx_resep]
    token_asli = [t for t in resep_asli if t != 0]

    if len(token_asli) == 0: continue

    # Kulkas positif (cocok)
    keep_ratio = random.uniform(0.4, 0.9)
    num_keep = max(1, int(len(token_asli) * keep_ratio))
    kulkas_simulasi = random.sample(token_asli, num_keep)
    kulkas_simulasi = kulkas_simulasi[:MAX_LEN] + [0] * (MAX_LEN - len(kulkas_simulasi))

    X1_list.append(kulkas_simulasi)
    X2_list.append(resep_asli)
    labels_list.append(1.0)

    # Kulkas negatif (zonk)
    idx_resep_neg = random.randint(0, num_recipes - 1)
    X1_list.append(kulkas_simulasi)
    X2_list.append(X_resep[idx_resep_neg])
    labels_list.append(0.0)

X1 = np.array(X1_list, dtype=np.float32)
X2 = np.array(X2_list, dtype=np.float32)
labels = np.array(labels_list, dtype=np.float32).reshape(-1, 1)

idx_shuffle = np.random.permutation(len(labels))
X1, X2, labels = X1[idx_shuffle], X2[idx_shuffle], labels[idx_shuffle]

##Arsitektur Functional API & Custom Loss (Quest 1, 2, 8)

In [ ]:
# --- CUSTOM COMPONENT 1: Custom Loss Function ---
class MAEAnnihilatorLoss(tf.keras.losses.Loss):
    def __init__(self, name="mae_annihilator_loss", **kwargs):
        super().__init__(name=name, **kwargs)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        mae = tf.abs(y_true - y_pred)
        extreme_penalty = tf.math.exp(mae * 5.0) - 1.0 # Hukuman keras agar MAE < 0.02
        return tf.reduce_mean(bce + extreme_penalty)

# --- FUNCTIONAL API TWO-TOWER ---
def buat_tower_sota(name, shared_embedding_layer):
    input_layer = layers.Input(shape=(MAX_LEN,), name=f"input_{name}")
    x = shared_embedding_layer(input_layer)

    # Custom Layer Lanjutan: Attention untuk konteks rasa
    attention_layer = layers.MultiHeadAttention(num_heads=2, key_dim=EMBED_DIM)
    x_attn = attention_layer(x, x)
    x = layers.Add()([x, x_attn])
    x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(TOWER_DIM, activation='linear')(x)
    output_layer = layers.Lambda(lambda val: tf.math.l2_normalize(val, axis=1))(x)

    return Model(inputs=input_layer, outputs=output_layer, name=f"tower_{name}")

shared_embedding = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)

input_kulkas = layers.Input(shape=(MAX_LEN,), name="Input_Kulkas")
input_resep  = layers.Input(shape=(MAX_LEN,), name="Input_Resep")

tower_kulkas = buat_tower_sota("kulkas", shared_embedding)
tower_resep  = buat_tower_sota("resep", shared_embedding)

vec_kulkas = tower_kulkas(input_kulkas)
vec_resep  = tower_resep(input_resep)

similarity_score = layers.Dot(axes=1)([vec_kulkas, vec_resep])
scaled_score = layers.Lambda(lambda x: x * 30.0)(similarity_score) # Trik Logit Scale SOTA
output_final = layers.Activation('sigmoid')(scaled_score)

model_sota = Model(inputs=[input_kulkas, input_resep], outputs=output_final)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

##Custom Training Loop

In [ ]:
# --- CUSTOM COMPONENT 2: Custom Callback ---
class EvaluasiSisaBisaCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1} | Val Acc: {logs['val_accuracy']*100:.1f}% | Val MAE: {logs['val_mae']:.4f}")

train_acc_metric = tf.keras.metrics.BinaryAccuracy(name="accuracy")
train_mae_metric = tf.keras.metrics.MeanAbsoluteError(name="mae")
val_acc_metric = tf.keras.metrics.BinaryAccuracy(name="val_accuracy")
val_mae_metric = tf.keras.metrics.MeanAbsoluteError(name="val_mae")

BATCH_SIZE = 256
val_split = int(0.2 * len(X1))
x1_val, x2_val, y_val = X1[:val_split], X2[:val_split], labels[:val_split]
x1_train, x2_train, y_train = X1[val_split:], X2[val_split:], labels[val_split:]

train_dataset = tf.data.Dataset.from_tensor_slices((x1_train, x2_train, y_train)).shuffle(1024).batch(BATCH_SIZE)
val_dataset = tf.data.Dataset.from_tensor_slices((x1_val, x2_val, y_val)).batch(BATCH_SIZE)

# Setup TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_writer = tf.summary.create_file_writer(log_dir + '/train')
val_writer = tf.summary.create_file_writer(log_dir + '/validation')

@tf.function
def train_step(x1_b, x2_b, y_b):
    with tf.GradientTape() as tape:
        preds = model_sota([x1_b, x2_b], training=True)
        loss = MAEAnnihilatorLoss()(y_b, preds)
    grads = tape.gradient(loss, model_sota.trainable_weights)
    optimizer.apply_gradients(zip(grads, model_sota.trainable_weights))
    train_acc_metric.update_state(y_b, preds)
    train_mae_metric.update_state(y_b, preds)
    return loss

@tf.function
def val_step(x1_b, x2_b, y_b):
    preds = model_sota([x1_b, x2_b], training=False)
    val_loss = MAEAnnihilatorLoss()(y_b, preds)
    val_acc_metric.update_state(y_b, preds)
    val_mae_metric.update_state(y_b, preds)
    return val_loss

cb = EvaluasiSisaBisaCallback()
print("Mulai Training GradientTape...")

for epoch in range(15): # 15 Epochs cukup
    for x1_b, x2_b, y_b in train_dataset:
        loss_val = train_step(x1_b, x2_b, y_b)

    with train_writer.as_default():
        tf.summary.scalar('accuracy', train_acc_metric.result(), step=epoch)
        tf.summary.scalar('mae', train_mae_metric.result(), step=epoch)

    for x1_b, x2_b, y_b in val_dataset:
        v_loss = val_step(x1_b, x2_b, y_b)

    v_acc, v_mae = val_acc_metric.result().numpy(), val_mae_metric.result().numpy()

    with val_writer.as_default():
        tf.summary.scalar('accuracy', v_acc, step=epoch)
        tf.summary.scalar('mae', v_mae, step=epoch)

    cb.on_epoch_end(epoch, {'val_accuracy': v_acc, 'val_mae': v_mae})

    # Early Stopping jika target Rubrik tercapai!
    if v_acc >= 0.85 and v_mae <= 0.02:
        print("✅ Target Akurasi 85% dan MAE 0.02 Tercapai!")
        break

    train_acc_metric.reset_state(); train_mae_metric.reset_state()
    val_acc_metric.reset_state(); val_mae_metric.reset_state()

##Export Model & Artifacts

In [ ]:
# Pre-Compute Embedding untuk performa secepat kilat di Backend
resep_embeddings_sota = tower_resep.predict(X_resep, verbose=0)
np.save("resep_embeddings_sota.npy", resep_embeddings_sota)

# Export Model & Tokenizer
model_sota.save("twotower_sisabisa_sota.keras")
tower_kulkas.save("tower_kulkas_sota.keras")

tokenizer_json = tokenizer.to_json()
with open('tokenizer_sota.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))

print("✅ Model siap produksi (.keras) berhasil disimpan!")

##Inference Sederhana & Fitur Generative AI

In [ ]:
# !pip install -q -U google-generativeai
import google.generativeai as genai
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Setup Generative AI (Ganti dengan API Key kamu)
genai.configure(api_key="API_KEY_GEMINI_KAMU_DISINI")
model_genai = genai.GenerativeModel('gemini-1.5-flash')

def rekomendasikan_serta_genai_sota(bahan_user: str, bahan_mau_basi: list):
    # =====================================================================
    # 1. TAHAP DEEP LEARNING (Mencari Resep via Vektorisasi Matriks Cepat)
    # =====================================================================
    bahan_list = [b.strip().lower() for b in bahan_user.split(', ')]
    kulkas_seq = tokenizer.texts_to_sequences([" ".join(bahan_list)])
    kulkas_pad = pad_sequences(kulkas_seq, maxlen=MAX_LEN, padding='post')

    # 🔥 PERBAIKAN 1: Prediksi vektor kulkas cukup 1x saja!
    emb_user = tower_kulkas.predict(kulkas_pad, verbose=0)[0]

    # 🔥 PERBAIKAN 1: Hitung 11.790 resep sekaligus pakai Matriks (Cuma 0.1 detik)
    raw_scores = np.dot(resep_embeddings_sota, emb_user) * 30.0
    all_scores = 1 / (1 + np.exp(-raw_scores))

    kandidat = []
    for idx, row in df.iterrows():
        bahan_resep_str = str(row['teks_kunci_sota']).lower()
        if bahan_mau_basi and not any(basi.lower() in bahan_resep_str for basi in bahan_mau_basi):
            continue # Hard-Filter: Harus pakai bahan basi!

        # Ambil skor langsung dari hasil matriks, TIDAK PERLU model.predict di dalam loop
        kandidat.append({
            'nama': row['nama_menu'],
            'bahan': row['bahan'],
            'skor': float(all_scores[idx]),
            'vektor': resep_embeddings_sota[idx]
        })

    kandidat = sorted(kandidat, key=lambda x: x['skor'], reverse=True)
    hasil_terpilih = [kandidat[0]] if kandidat else []

    # Algoritma MMR (Mencari 3 Resep dengan gaya beda)
    for kand in kandidat[1:]:
        if len(hasil_terpilih) >= 3: break
        v_kand = np.array(kand['vektor']).reshape(1, -1)
        if not any(cosine_similarity(v_kand, np.array(t['vektor']).reshape(1, -1))[0][0] > 0.75 for t in hasil_terpilih):
            hasil_terpilih.append(kand)

    print("\n🍽️ REKOMENDASI AI SISA-BISA 🍽️")
    print("=" * 45)
    for i, item in enumerate(hasil_terpilih, 1):
        print(f"{i}. {item['nama']} (Kecocokan: {item['skor']*100:.1f}%)")

    # =====================================================================
    # 2. TAHAP GENERATIVE AI (Simulasi Logic Backend)
    # =====================================================================
    if hasil_terpilih:
        resep_utama = hasil_terpilih[0]
        print(f"\n✨ [HYBRID AI DEMO] Meminta Langkah & Gizi untuk '{resep_utama['nama']}' ke Gemini...")

        # 🔥 PERBAIKAN 2: Sesuaikan prompt dengan tujuan aslimu!
        prompt = f"""
        Saya ingin memasak {resep_utama['nama']} dengan bahan: {resep_utama['bahan']}.
        Tolong berikan:
        1. Langkah-langkah memasak yang berurutan.
        2. Estimasi Nutrition Facts (Kalori, Protein, Lemak) per porsi.
        """

        try:
            respons = model_genai.generate_content(prompt)
            print("-" * 45)
            print(respons.text)
            print("-" * 45)
        except Exception as e:
            print(f"⚠️ API Key Generative AI belum di-set atau kuota habis. Error: {e}")

# JALANKAN INFERENCE
rekomendasikan_serta_genai_sota(
    bahan_user="daging sapi, kecap manis, bawang merah, cabai",
    bahan_mau_basi=["daging sapi"]
)